In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
os.listdir()

['.ipynb_checkpoints',
 'cartera_variable_respuesta_candidato.csv',
 'dataset_candidato.csv',
 'generacion_y.ipynb',
 'Prueba_Correo_Instrucciones_DS_Risk_Analytics.docx',
 'respuesta.csv',
 'Revision variables.ipynb']

# Construccion de la Y

In [3]:
##como normalmente se ve la información de cartera en la operación real
real=pd.read_csv("cartera_variable_respuesta_candidato.csv",engine='c')
real

,application_id,fecha_apertura,fecha_corte,altura_mora,saldo_capital
0,CAND-00001,2024-03-20,2024-04-19,0,428989.0
1,CAND-00001,2024-03-20,2024-05-19,0,386445.0
2,CAND-00001,2024-03-20,2024-06-18,0,347446.0
3,CAND-00001,2024-03-20,2024-07-18,0,314018.0
4,CAND-00002,2024-08-22,2024-09-21,0,784934.0
...,...,...,...,...,...
18164,CAND-PH0122,2024-07-19,2024-08-18,0,496894.0
18165,CAND-PH0122,2024-07-19,2024-09-17,0,465716.0
18166,CAND-PH0122,2024-07-19,2024-10-17,30,450993.0
18167,CAND-PH0122,2024-07-19,2024-11-16,60,434932.0


In [4]:
## es necesario corregir la fecha de apertura no esta en el mismo formato
real["fecha_apertura"].str.split("-",expand=True)[0].value_counts()

2024    17435
2025      652
16         10
22         10
02          8
18          8
05          6
14          5
13          5
15          4
19          4
30          4
07          3
10          3
29          3
28          3
04          3
08          3
Name: 0, dtype: int64

In [5]:
real['fecha_apertura'] = pd.to_datetime(real['fecha_apertura'],infer_datetime_format=True)
real

C:\Users\jsaj_\AppData\Local\Temp\ipykernel_3032\3225034706.py:1: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  real['fecha_apertura'] = pd.to_datetime(real['fecha_apertura'],infer_datetime_format=True)


,application_id,fecha_apertura,fecha_corte,altura_mora,saldo_capital
0,CAND-00001,2024-03-20,2024-04-19,0,428989.0
1,CAND-00001,2024-03-20,2024-05-19,0,386445.0
2,CAND-00001,2024-03-20,2024-06-18,0,347446.0
3,CAND-00001,2024-03-20,2024-07-18,0,314018.0
4,CAND-00002,2024-08-22,2024-09-21,0,784934.0
...,...,...,...,...,...
18164,CAND-PH0122,2024-07-19,2024-08-18,0,496894.0
18165,CAND-PH0122,2024-07-19,2024-09-17,0,465716.0
18166,CAND-PH0122,2024-07-19,2024-10-17,30,450993.0
18167,CAND-PH0122,2024-07-19,2024-11-16,60,434932.0


In [6]:
### cantidad de veces que aprece una persona en pagos
real["application_id"].value_counts().reset_index()["application_id"].value_counts()

4    1031
3    1021
5    1018
6     982
Name: application_id, dtype: int64

In [7]:
print(f"fecha maxima {real['fecha_apertura'].max().date()}")
print(f"fecha minima {real['fecha_apertura'].min().date()}")

fecha maxima 2025-10-01
fecha minima 2024-01-04


In [8]:
real['fecha_apertura'].max().date()-real['fecha_apertura'].min().date()

datetime.timedelta(days=636)

In [9]:
# Número de corte de cada fila dentro de su crédito (1, 2, 3, ...)
real = real.sort_values(["application_id", "fecha_corte"])
real["n_corte"] = real.groupby("application_id").cumcount() + 1
 
# =============================================================
# 2. Resumen por crédito
# =============================================================
creditos = real.groupby("application_id").agg(
    fecha_apertura=("fecha_apertura", "first"),
    n_cortes=("n_corte", "max"),
    mora_max=("altura_mora", "max"),
    mora_ultima=("altura_mora", "last"),
    saldo_inicial=("saldo_capital", "first"),
    saldo_final=("saldo_capital", "last"),
)
creditos["cohorte"] = creditos["fecha_apertura"].dt.to_period("M")
creditos["pct_saldo_final"] = creditos["saldo_final"] / creditos["saldo_inicial"]
 
print("\nCréditos:", len(creditos))
print("\nCréditos por número de cortes:")
print(creditos["n_cortes"].value_counts().sort_index())
 
# ¿Los créditos terminan pagados o se dejan de observar?
print("\n% del saldo inicial que queda en el último corte:")
print(creditos.groupby("n_cortes")["pct_saldo_final"].mean().round(3))


Créditos: 4052

Créditos por número de cortes:
3    1021
4    1031
5    1018
6     982
Name: n_cortes, dtype: int64

% del saldo inicial que queda en el último corte:
n_cortes
3    0.864
4    0.792
5    0.722
6    0.652
Name: pct_saldo_final, dtype: float64


In [10]:
# =============================================================
# 3. Tabla de roll rates (transición de un corte al siguiente)
# =============================================================
real["tramo"] = pd.cut(real["altura_mora"], [-1, 0, 30, 60, np.inf],
                       labels=["0", "30", "60", "90+"])
real["tramo_siguiente"] = real.groupby("application_id")["tramo"].shift(-1)
 
print("\nRoll rates (conteos):")
print(pd.crosstab(real["tramo"], real["tramo_siguiente"], margins=True))
print("\nRoll rates (% por fila):")
print((pd.crosstab(real["tramo"], real["tramo_siguiente"], normalize="index") * 100).round(1))


Roll rates (conteos):
tramo_siguiente      0    30   60  90+    All
tramo                                        
0                10684  1191    0    0  11875
30                 710     0  658    0   1368
60                   0    51    0  437    488
90+                  0     0   28  358    386
All              11394  1242  686  795  14117

Roll rates (% por fila):
tramo_siguiente     0    30    60   90+
tramo                                  
0                90.0  10.0   0.0   0.0
30               51.9   0.0  48.1   0.0
60                0.0  10.5   0.0  89.5
90+               0.0   0.0   7.3  92.7


In [11]:
# =============================================================
# 4. Tasa de default según umbral ("alguna vez", todo lo observado)
# =============================================================
umbrales = {"≥30": 30, "≥60": 60, "≥90": 90, ">90": 91, "≥120": 120}
tabla_umbral = pd.DataFrame({
    "umbral": list(umbrales.keys()),
    "defaults": [(creditos["mora_max"] >= u).sum() for u in umbrales.values()],
    "tasa_%": [round((creditos["mora_max"] >= u).mean() * 100, 1) for u in umbrales.values()],
})
print("\nTasa de default por umbral (4.052 créditos):")
print(tabla_umbral.to_string(index=False))


Tasa de default por umbral (4.052 créditos):
umbral  defaults  tasa_%
   ≥30      1509    37.2
   ≥60       632    15.6
   ≥90       422    10.4
   >90       200     4.9
  ≥120         9     0.2


In [12]:
# =============================================================
# 5. Sesgo por número de cortes observados
# =============================================================
for nombre, u in [("d30", 30), ("d60", 60), ("d90", 90)]:
    creditos[nombre] = (creditos["mora_max"] >= u).astype(int)
 
print("Tasa 'alguna vez' por número de cortes (%):")
print((creditos.groupby("n_cortes")[["d30", "d60", "d90"]].mean() * 100).round(1))

Tasa 'alguna vez' por número de cortes (%):
           d30   d60   d90
n_cortes                  
3         30.7  14.6   7.2
4         34.9  14.5   8.4
5         39.6  16.2  12.7
6         44.1  17.1  13.4


In [13]:
# =============================================================
# 6. Ventana fija: mora máxima en los primeros k cortes
# =============================================================
filas = []
for k in [3, 4, 5, 6]:
    mora_k = real[real["n_corte"] <= k].groupby("application_id")["altura_mora"].max()
    elegibles = creditos.index[creditos["n_cortes"] >= k]   # solo créditos con k cortes completos
    m = mora_k.loc[elegibles]
    filas.append({"ventana_cortes": k, "dias": k * 30, "creditos": len(elegibles),
                  "tasa_30_%": round((m >= 30).mean() * 100, 1),
                  "tasa_60_%": round((m >= 60).mean() * 100, 1),
                  "tasa_90_%": round((m >= 90).mean() * 100, 1)})
print("\nTasa con ventana fija:")
print(pd.DataFrame(filas).to_string(index=False))


Tasa con ventana fija:
 ventana_cortes  dias  creditos  tasa_30_%  tasa_60_%  tasa_90_%
              3    90      4052       28.5       10.5        5.1
              4   120      3031       34.4       13.2        8.1
              5   150      2000       39.4       15.2       11.9
              6   180       982       44.1       17.1       13.4


In [14]:
# =============================================================
# 7. Tasa por cohorte de apertura (el tip de las fechas)
# =============================================================
por_cohorte = creditos.groupby("cohorte").agg(
    creditos=("d90", "size"),
    tasa_30=("d30", "mean"), tasa_60=("d60", "mean"), tasa_90=("d90", "mean"))
por_cohorte[["tasa_30", "tasa_60", "tasa_90"]] = (por_cohorte[["tasa_30", "tasa_60", "tasa_90"]] * 100).round(1)
print("\nTasa por cohorte (%):")
print(por_cohorte)
 
creditos["periodo"] = np.where(creditos["fecha_apertura"] >= "2024-11-01",
                               "nov24-ene25", "ene24-oct24")
print("\nCohortes antiguas vs recientes (%):")
print((creditos.groupby("periodo")[["d30", "d60", "d90"]].mean() * 100).round(1))
 


Tasa por cohorte (%):
         creditos  tasa_30  tasa_60  tasa_90
cohorte                                     
2024-01       182     34.1     12.6      9.3
2024-02       329     35.9     15.2     10.3
2024-03       341     33.7     16.1     11.1
2024-04       330     36.1     14.8      9.4
2024-05       383     37.1     13.3      9.1
2024-06       332     35.2     13.3      9.6
2024-07       346     37.0     15.3     10.7
2024-08       337     35.6     14.8      8.9
2024-09       319     33.2     12.2      8.2
2024-10       323     37.5     13.9      8.4
2024-11       325     42.2     22.2     14.5
2024-12       355     44.2     19.2     13.2
2025-01       149     44.3     21.5     13.4
2025-10         1    100.0    100.0    100.0

Cohortes antiguas vs recientes (%):
              d30   d60   d90
periodo                      
ene24-oct24  35.6  14.2   9.5
nov24-ene25  43.5  20.8  13.9


In [15]:
# =============================================================
# 8. Variable Y final: ≥90 días en los primeros 4 cortes
# =============================================================
VENTANA, UMBRAL = 4, 90
mora_ventana = real[real["n_corte"] <= VENTANA].groupby("application_id")["altura_mora"].max()
creditos["default"] = (mora_ventana >= UMBRAL).astype(int)
creditos["valido"] = creditos["n_cortes"] >= VENTANA
 
y = creditos.loc[creditos["valido"], ["fecha_apertura", "cohorte", "default"]].reset_index()
print(f"\nY final: {len(y)} créditos, tasa de default = {y['default'].mean():.1%}")
 
# Alternativa de robustez: ≥60 días en los primeros 3 cortes (todos los créditos)
mora_3 = real[real["n_corte"] <= 3].groupby("application_id")["altura_mora"].max()
print(f"Alternativa ≥60 en 3 cortes: {len(mora_3)} créditos, tasa = {(mora_3 >= 60).mean():.1%}")


Y final: 3031 créditos, tasa de default = 8.1%
Alternativa ≥60 en 3 cortes: 4052 créditos, tasa = 10.5%


In [16]:
y.to_csv("respuesta.csv",sep="|",index=False)